# Acts 1–2: an agent, then its harness

First run an SDK agent with instructions and tools. Then add named layers and compare context selection.

**Runtime:** Python 3.13+ is required by this repository. The setup checks the active notebook kernel. Use a compatible Jupyter/Colab runtime; Colab runtime compatibility has not been validated here. Installing a separate Python executable does not switch the kernel. Cells marked **Live** call the model API. No real orders or emails are sent.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

root = Path.cwd()
if not (root / "formaggio" / "config.py").exists():
    if (root.parent / "formaggio" / "config.py").exists():
        root = root.parent
    else:
        root = root / "agent-harness"
        if not root.exists():
            subprocess.run(["git", "clone", "https://github.com/dlops-io/agent-harness.git", str(root)], check=True)
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
print("Repository:", root)

In [ ]:
import tomllib

if sys.version_info < (3, 13):
    raise RuntimeError("Select a Python 3.13+ notebook kernel before continuing.")
project = tomllib.loads(Path("pyproject.toml").read_text())
subprocess.run([sys.executable, "-m", "pip", "install", *project["project"]["dependencies"]], check=True)
subprocess.run([sys.executable, "-m", "scripts.check_sdk"], check=True)

The next cell obtains a model credential without saving it in the notebook. Use the repository revision containing this layer refactor. Dependency setup reads the project's pinned direct dependencies; Docker/uv remains the existing locked environment path.

In [ ]:
from getpass import getpass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API key: ")

## Shared lesson setup

The customer request is a confirmed scenario. Shop tools use catalog and policy data from the repository. Both the bare and harness runs use the same agent definition: instructions, five tools, and the `AgentReply` schema.

In [ ]:
from agent_framework import Agent
from agent_framework.openai import OpenAIChatClient
from openai import AsyncOpenAI

from acts.act1_agent import build_agent
from acts.act2_context import build_act2
from cli import print_agent_result
from formaggio.config import MODEL
from formaggio.agents.context import customer_ask, customer_message, instructions, load_scenario
from formaggio.agents.harness import Harness
from formaggio.agents.layers import Trace, Budget, Context, CartCheck
from formaggio.agents.runtime import FUNCTION_LIMITS, MODEL_OPTIONS
from formaggio.agents.tools import build_tools
from formaggio.operations.observability import Recorder
from formaggio.shop.data_models import AgentReply
from formaggio.shop.store import Store

DB_PATH = Path("outputs/notebook.sqlite")
store, request = Store(), load_scenario("standard")
print("👤 Customer ask:\n" + customer_ask(request))

## Act 1 — Run the agent (Live)

This cell is the bare agent: create it, run it, and read the response. `build_agent()` in the act module packages this same definition for reuse below.

The SDK still provides its tool loop and configured execution limits. Shop tools retain their domain checks. This run does not write a database trace or independently assess the final cart. `preview_order` returns a proposal; it cannot place an order.

In [ ]:
async with AsyncOpenAI(timeout=45, max_retries=0) as api:
    client = OpenAIChatClient(model=MODEL, async_client=api, function_invocation_configuration=FUNCTION_LIMITS)
    agent = Agent(
        client=client,
        name="FormaggioAssistant",
        instructions=instructions(store),
        tools=build_tools(store, request, None, None),
        default_options={**MODEL_OPTIONS, "response_format": AgentReply},
    )
    response = await agent.run(customer_message(request))
print(response.value.message)

## Act 1 — Add the harness (Live)

The layers make four concepts visible: tracing, execution budgets, context selection, and independent final-cart assessment. The harness also owns the run lifecycle, fresh invocation state, and required tool audit. `.run()` is the same explicit entry point for every configuration.

In [ ]:
harness = (
    Harness(build_agent, model=MODEL)
    .add(Trace())
    .add(Budget(model_calls=8, tool_calls=20, seconds=120))
    .add(Context("basic"))
    .add(CartCheck())
)
with Recorder(DB_PATH) as recorder:
    result = await harness.run(recorder, progress=print)
print_agent_result(result)

**Checkpoint:** Which statements came from the model, and which came from the independent cart check? Does a valid proposal place an order?

## Remove one feature (Live)

`.without()` returns another configuration. Removing `CartCheck` leaves the response intact and marks the final check as `not_run`. It does not remove rules enforced inside shop tools. Each run gets fresh state, so live responses may differ.

In [ ]:
without_check = harness.without("cart_check")
with Recorder(DB_PATH) as recorder:
    result_without_check = await without_check.run(recorder, progress=print)
print_agent_result(result_without_check)

Try `harness.without("trace")` to remove detailed telemetry. Counters and required tool audit remain. Removing `Budget` retains the SDK's recorded 40-iteration fallback, but removes the exact model/tool caps and overall run timeout. Use the offline tests to explore looping behavior.

## Act 2 — Compare context (Live)

Both runs use the same agent definition and confirmed request. The enriched run supplies selected catalog context and this customer's saved preferences. Each run starts a fresh session and inventory.

In [ ]:
context_results = {}
for mode in ("basic", "enriched"):
    lesson = build_act2(mode=mode, model=MODEL)
    with Recorder(DB_PATH) as recorder:
        context_results[mode] = await lesson.run(recorder, progress=print)
    print_agent_result(context_results[mode])

**Checkpoint:** Compare model/tool calls, the context sources, and final-cart reports. A different live response alone does not establish better quality.

## Inspect evidence (Offline)

Print event names from the last recorded run. The CLI's `--inspect-run RUN_ID` shows full recorded payloads. `ConsoleProgress(show_json=True)` enables detailed JSON previews during notebook runs.

In [ ]:
with Recorder(DB_PATH) as recorder:
    run_id = context_results["enriched"]["run_id"]
    for event in recorder.timeline(run_id):
        print(event["event_type"])

For deterministic regression checks, run `python -m tests` from the repository. These use local scripted responses, including layer-removal, initialization-failure, cancellation, and overlapping-run cases. The named-layer API currently applies to Acts 1–2; later acts retain their existing implementations.